In [ ]:
import pandas as pd
from collections import OrderedDict

# Read all sheets from the Excel file into a dictionary
xls = pd.read_excel('MASTER_DF_final.xlsx', sheet_name=None)

# View all available sheet names
sheet_names = list(xls.keys())
print("Sheets found:", sheet_names)

# Dictionary to store OrderedDicts for each trial
ordered_data_by_sheet = OrderedDict()

# Access each sheet individually and convert rows to OrderedDict
df_trial1 = xls.get('Trial1')
df_trial2 = xls.get('Trial2')
df_trial3 = xls.get('Trial3')
df_trial4 = xls.get('Trial4')
df_trial5 = xls.get('Trial5')
df_trial6 = xls.get('Trial6')
df_trial7 = xls.get('Trial7')
df_trial8 = xls.get('Trial8')
df_trial9 = xls.get('Trial9')
df_trial10 = xls.get('Trial10')
df_trial11 = xls.get('Trial11')

# Manually list trials you want to convert
trial_dfs = {
    'Trial1': df_trial1,
    'Trial2': df_trial2,
    'Trial3': df_trial3,
    'Trial4': df_trial4,
    'Trial5': df_trial5,
    'Trial6': df_trial6,
    'Trial7': df_trial7,
    'Trial8': df_trial8,
    'Trial9': df_trial9,
    'Trial10': df_trial10,
    'Trial11': df_trial11
}

# Convert each DataFrame row into OrderedDict and store
for sheet_name, df in trial_dfs.items():
    if df is not None:
        records = [OrderedDict(row.items()) for _, row in df.iterrows()]
        ordered_data_by_sheet[sheet_name] = records


'''
# Optional: print a preview
for sheet, records in ordered_data_by_sheet.items():
    print(f"\nSheet: {sheet}")
    for i, row in enumerate(records[:2]):  # preview first 2 rows per sheet
        print(f" Row {i+1}: {row}")
'''

In [ ]:
# Master excel sheet
master = OrderedDict()

for trial_key in ordered_data_by_sheet:
    trial_data = ordered_data_by_sheet[trial_key]
    if trial_key not in master:
        master[trial_key] = OrderedDict()

    for row in trial_data:
        if row.get('Treatment') != 'T4':
            continue

        date = row.get('Date')
        plot_name = row.get('Name of plot')

        if not date or not plot_name:
            continue

        if date not in master[trial_key]:
            master[trial_key][date] = OrderedDict()

        # Build the required output values
        values = OrderedDict({
            'DAT': row.get('DAT'),
            'PDI_SB_new': row.get('PDI_SB_new'),
            'new_thrips': row.get('new_thrips'),
            'PDI_PB_new': row.get('PDI_PB_new'),
            'PDI_AN_new': row.get('PDI_AN_new')
        })

        master[trial_key][date][plot_name] = values

print (master['Trial1'])
#print ("\n\n")
#print (master['Trial5']['20/10/2023'])

In [ ]:
# Weather master data
# Read all sheets from the Excel file into a dictionary
xls = pd.read_excel('weather_master_data.xlsx', sheet_name=None)

# View all available sheet names
sheet_names = list(xls.keys())
print("Sheets found:", sheet_names)

# Dictionary to store OrderedDicts for each trial
ordered_data_weather = OrderedDict()

# Access each sheet individually and convert rows to OrderedDict
df_trial1 = xls.get('Trial1')
df_trial2 = xls.get('Trial2')
df_trial3 = xls.get('Trial3')
df_trial4 = xls.get('Trial4')
df_trial5 = xls.get('Trial5')
df_trial6 = xls.get('Trial6')
df_trial7 = xls.get('Trial7')
df_trial8 = xls.get('Trial8')
df_trial9 = xls.get('Trial9')
df_trial10 = xls.get('Trial10')
df_trial11 = xls.get('Trial11')

# Manually list trials you want to convert
trial_dfs = {
    'Trial1': df_trial1,
    'Trial2': df_trial2,
    'Trial3': df_trial3,
    'Trial4': df_trial4,
    'Trial5': df_trial5,
    'Trial6': df_trial6,
    'Trial7': df_trial7,
    'Trial8': df_trial8,
    'Trial9': df_trial9,
    'Trial10': df_trial10,
    'Trial11': df_trial11
}

# Convert each DataFrame row into OrderedDict and store
for sheet_name, df in trial_dfs.items():
    if df is not None:
        records = [OrderedDict(row.items()) for _, row in df.iterrows()]
        ordered_data_weather[sheet_name] = records

'''
# Optional: print a preview
for sheet, records in ordered_data_weather.items():
    print(f"\nSheet: {sheet}")
    for i, row in enumerate(records[:2]):  # preview first 2 rows per sheet
        print(f" Row {i+1}: {row}")
'''

In [ ]:
from collections import OrderedDict
from copy import deepcopy

weather_eda = OrderedDict()

for trial_key in ordered_data_weather:  # assuming your input dict is called weather_data_by_sheet
    trial_data = ordered_data_weather[trial_key]  # list of rows (OrderedDict)

    if trial_key not in weather_eda:
        weather_eda[trial_key] = OrderedDict()

    for row in trial_data:
        date = row.get('Date')
        if not date:
            continue

        # Store a deep copy of the entire row (excluding the Date key if you prefer)
        weather_eda[trial_key][date] = deepcopy(row)

print (weather_eda['Trial1'])

In [ ]:
from collections import OrderedDict, defaultdict
import re

# Grouping function for a single day's data
def group_data(entry_dict):
    grouped = defaultdict(OrderedDict)
    for key, value in entry_dict.items():
        if re.match(r'^B[1-5]', key):
            block = key[:2]  # 'B1', 'B2', etc.
            grouped[block][key] = value
        elif "prevday" in key:
            grouped["prev_day"][key] = value
        elif "last_7days" in key:
            grouped["last_7_days"][key] = value
        else:
            grouped["others"][key] = value
    return grouped

# Main loop: group all trials and dates
grouped_weather = OrderedDict()

for trial_name, trial_data in weather_eda.items():
    grouped_weather[trial_name] = OrderedDict()
    for date, daily_data in trial_data.items():
        grouped_weather[trial_name][date] = group_data(daily_data)

# Optional: pretty print
from pprint import pprint
pprint(grouped_weather['Trial9']['30/12/2023'])


In [ ]:
# ALL Data together
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
from datetime import datetime

# Set journal-style aesthetics
sns.set(style="whitegrid", context="notebook", font_scale=1.2)
color_palette = sns.color_palette("deep")

# Data structure
region_data = {'R1T4': [], 'R2T4': [], 'R3T4': []}
metrics = ['PDI_SB_new', 'new_thrips', 'PDI_PB_new', 'PDI_AN_new']

# Replace with your actual master OrderedDict
# master = OrderedDict({ ... })

# Extract and organize data
for trial_name, trial_data in master.items():
    for date_str, date_data in trial_data.items():
        for region, values in date_data.items():
            if region in region_data:
                entry = {
                    'Date': datetime.strptime(date_str, "%d/%m/%Y"),
                }
                for metric in metrics:
                    entry[metric] = values.get(metric, 0)
                region_data[region].append(entry)

# Create 3x4 subplots (3 regions × 4 metrics)
fig, axes = plt.subplots(3, 4, figsize=(20, 12), sharex='col')
fig.subplots_adjust(hspace=0.4, wspace=0.3)

# Plotting
for row_idx, (region, data) in enumerate(region_data.items()):
    data_sorted = sorted(data, key=lambda x: x['Date'])
    dates = [entry['Date'] for entry in data_sorted]

    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx][col_idx]
        values = [entry[metric] for entry in data_sorted]
        ax.plot(dates, values, marker='o', linestyle = '', color=color_palette[col_idx])
        ax.set_title(f"{region} - {metric}", fontsize=11)
        ax.set_ylabel("Value")
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True)

axes[-1][0].set_xlabel("Date")
axes[-1][1].set_xlabel("Date")
axes[-1][2].set_xlabel("Date")
axes[-1][3].set_xlabel("Date")

fig.suptitle("Metric-wise Trends Across Regions", fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import OrderedDict
from datetime import datetime

# Journal style
sns.set(style="whitegrid", context="notebook", font_scale=1.2)
color_palette = sns.color_palette("deep")

# Metrics and Regions
metrics = ['PDI_SB_new', 'new_thrips', 'PDI_PB_new', 'PDI_AN_new']
regions = ['R1T4', 'R2T4', 'R3T4']

# Loop through each trial
for trial_name, trial_data in master.items():
    # Structure to hold data region-wise
    region_data = {region: [] for region in regions}

    # Extract data for this trial
    for date_str, date_data in trial_data.items():
        date_obj = datetime.strptime(date_str, "%d/%m/%Y")
        for region in regions:
            if region in date_data:
                entry = {'Date': date_obj}
                for metric in metrics:
                    entry[metric] = date_data[region].get(metric, 0)
                region_data[region].append(entry)
    
    # ✅ Create 3x4 subplot grid here
    fig, axes = plt.subplots(3, 4, figsize=(20, 12))  # <--- MISSING LINE
    fig.suptitle(f"Trial: {trial_name} - Metric Trends", fontsize=16, weight='bold', y=1.02)
    fig.subplots_adjust(hspace=0.4, wspace=0.3)

    # Plot data
    for row_idx, region in enumerate(regions):
        data_sorted = sorted(region_data[region], key=lambda x: x['Date'])
        dates = [entry['Date'] for entry in data_sorted]

        for col_idx, metric in enumerate(metrics):
            ax = axes[row_idx][col_idx]
            values = [entry[metric] for entry in data_sorted]
            ax.plot(dates, values, marker='o', color=color_palette[col_idx])
            ax.set_title(f"{region} - {metric}", fontsize=11)
            ax.set_ylabel("Value")
            ax.set_xticks(dates)
            ax.set_xticklabels([d.strftime('%d-%b') for d in dates], rotation=45)
            ax.grid(True)

    # Set X-axis label only for the bottom row
    for col in range(4):
        axes[2][col].set_xlabel("Date")

    # Final layout and display
    plt.tight_layout()
    plt.show()


In [ ]:
from collections import OrderedDict
from datetime import datetime

# Assume: master, regions, metrics, grouped_weather are already defined
split_region_data = OrderedDict()

for trial_name, trial_data in master.items():
    split_region_data[trial_name] = OrderedDict()

    for region in regions:
        split_region_data[trial_name][region] = OrderedDict()  # each region is a dict of blocks

    for date_str, date_data in trial_data.items():
        date_obj = datetime.strptime(date_str, "%d/%m/%Y")

        # Fetch grouped weather for this trial and date
        weather_blocks = grouped_weather.get(trial_name, {}).get(date_str, {})  # e.g. {'B1': {...}, 'B2': {...}}

        for region in regions:
            if region in date_data:
                for metric in metrics:
                    metric_value = date_data[region].get(metric, 0)

                    for block_name, block_values in weather_blocks.items():
                        # Initialize block list if not already
                        if block_name not in split_region_data[trial_name][region]:
                            split_region_data[trial_name][region][block_name] = []
                        if block_name == 'others':
                            #print(date_obj, metric_value)
                            entry = {'Date': date_obj, metric: metric_value}
                            entry['RHdelta'] = block_values['RHdelta']
                            split_region_data[trial_name][region][block_name].append(entry)
                        else:
                            # Create entry with date, metric value and all block values
                            entry = {'Date': date_obj, metric: metric_value}
                            #print(date_obj, metric_value)
                            entry.update(block_values)  # Add all weather block values
                            split_region_data[trial_name][region][block_name].append(entry)

                #break  # process each region only once per date
print(split_region_data['Trial1']['R3T4']['others'])

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime
import math

# Define the metrics to be plotted
metrics_to_plot = ["new_thrips", "PDI_SB_new", "PDI_PB_new", "PDI_AN_new"]

# Access your data
entries = split_region_data['Trial1']['R3T4']['others']

# Extract all unique dates
available_dates = sorted(set(e['Date'] for e in entries if isinstance(e.get('Date'), datetime)))

# Setup subplot grid
n_cols = 4
n_rows = math.ceil(len(metrics_to_plot) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharex=False)
axes = axes.flatten()  # Make axes 1D list for easy indexing

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]

    # Filter entries that contain both the metric and RHdelta
    filtered_entries = [
        e for e in entries
        if metric in e and 'RHdelta' in e and e['Date'] in available_dates
    ]

    if not filtered_entries:
        ax.set_title(f"{metric}\nNo data")
        ax.axis("off")
        continue

    # Sort entries by date
    filtered_entries.sort(key=lambda x: x['Date'])

    # Extract data
    dates = [e['Date'].strftime('%d-%b') for e in filtered_entries]
    metric_values = [e[metric] for e in filtered_entries]
    rhdelta_values = [e['RHdelta'] for e in filtered_entries]

    # Plotting
    ax.plot(dates, metric_values, marker='o', label=metric, color='blue')
    ax.plot(dates, rhdelta_values, marker='s', label='RHdelta', color='orange')

    ax.set_xlabel("Date")
    ax.set_ylabel("Value")
    # Set y-axis limits (adjust as needed)
    ymin = min(min(metric_values), min(rhdelta_values)) - 1
    ymax = max(max(metric_values), max(rhdelta_values)) + 1
    ax.set_ylim(ymin, 100)
    ax.set_title(f"{metric} vs RHdelta")
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True)
    ax.legend()

# Turn off any unused subplots
for j in range(len(metrics_to_plot), len(axes)):
    axes[j].axis("off")

plt.suptitle("Trial1 - R3T4 - Block: others", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime
import math

# Define the metrics to be plotted
metrics_to_plot = ["new_thrips", "PDI_SB_new", "PDI_PB_new", "PDI_AN_new"]
regions = ['R1T4', 'R2T4', 'R3T4']
trial_name = 'Trial4'
block_name = 'others'

for region in regions:
    entries = split_region_data[trial_name][region][block_name]

    # Extract all unique dates for sorting and consistency
    available_dates = sorted(set(e['Date'] for e in entries if isinstance(e.get('Date'), datetime)))

    # Setup subplot grid
    n_cols = 4
    n_rows = math.ceil(len(metrics_to_plot) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharex=False)
    axes = axes.flatten()

    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]

        # Filter entries with valid metric and RHdelta
        filtered_entries = [
            e for e in entries
            if metric in e and 'RHdelta' in e and e['Date'] in available_dates
        ]

        if not filtered_entries:
            ax.set_title(f"{metric}\nNo data")
            ax.axis("off")
            continue

        # Sort entries by date
        filtered_entries.sort(key=lambda x: x['Date'])

        # Extract data
        dates = [e['Date'].strftime('%d-%b') for e in filtered_entries]
        metric_values = [e[metric] for e in filtered_entries]
        rhdelta_values = [e['RHdelta'] for e in filtered_entries]

        # Plotting
        ax.plot(dates, metric_values, marker='o', label=metric, color='blue')
        ax.plot(dates, rhdelta_values, marker='s', label='RHdelta', color='orange')

        ax.set_xlabel("Date")
        ax.set_ylabel("Value")
        # Set y-axis limits (adjust as needed)
        ymin = min(min(metric_values), min(rhdelta_values)) - 1
        ymax = max(max(metric_values), max(rhdelta_values)) + 1
        ax.set_ylim(ymin, 100)
        ax.set_title(f"{metric} vs RHdelta")
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True)
        ax.legend()

    # Hide unused subplots
    for j in range(len(metrics_to_plot), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"{trial_name} - {region} - Block: {block_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

# Disease metrics
disease_metrics = ["new_thrips", "PDI_SB_new", "PDI_PB_new", "PDI_AN_new"]
block_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'last_7_days', 'others', 'prev_day']

# Loop through each trial
for trial_name, trial_data in split_region_data.items():
    print(f"\n🔍 Plotting data for trial: {trial_name}")

    for region in trial_data:
        for block in block_names:
            entries = split_region_data[trial_name][region][block]

            block_vars = sorted(set(
                k for e in entries for k in e
                if k not in disease_metrics and k != 'Date'
            ))

            if not entries or not block_vars:
                print(f"⚠️ No valid data for {trial_name} - {region} - {block}")
                continue

            fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=False)
            axes = axes.flatten()

            for i, disease_metric in enumerate(disease_metrics):
                ax = axes[i]
                all_dates = []

                for block_var in block_vars:
                    filtered = [
                        e for e in entries
                        if block_var in e and disease_metric in e and isinstance(e.get('Date'), datetime)
                    ]
                    if not filtered:
                        continue

                    filtered.sort(key=lambda x: x['Date'])
                    dates = [e['Date'] for e in filtered]
                    y_block = [e[block_var] for e in filtered]
                    ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
                    all_dates.extend(dates)

                disease_filtered = [
                    e for e in entries if disease_metric in e and isinstance(e.get('Date'), datetime)
                ]
                if disease_filtered:
                    disease_filtered.sort(key=lambda x: x['Date'])
                    dates_disease = [e['Date'] for e in disease_filtered]
                    y_disease = [e[disease_metric] for e in disease_filtered]

                    ax2 = ax.twinx()
                    ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=disease_metric)
                    ax2.set_ylabel(disease_metric, color='red')
                    ax2.tick_params(axis='y', labelcolor='red')
                    ax2.set_ylim(top=100)

                ax.set_title(f"{disease_metric} vs Block Vars")
                ax.set_ylabel("Block Var Value")
                ax.tick_params(axis='x', rotation=45)
                ax.grid(True)

                if all_dates:
                    ax.set_xticks(sorted(set(all_dates)))
                    ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

                ax.legend(fontsize='small', loc='upper left')

            plt.suptitle(f"{trial_name} | Region: {region} | Block: {block}", fontsize=16)
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()


In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

# Disease metrics
disease_metrics = ["new_thrips", "PDI_SB_new", "PDI_PB_new", "PDI_AN_new"]
block_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'last_7_days', 'others', 'prev_day']

# Loop through each trial
for trial_name, trial_data in split_region_data.items():
    print(f"\n🔍 Plotting data for trial: {trial_name}")

    for region in trial_data:
        for block in block_names:
            entries = split_region_data[trial_name][region][block]

            block_vars = sorted(set(
                k for e in entries for k in e
                if k not in disease_metrics and k != 'Date'
            ))

            if not entries or not block_vars:
                print(f"⚠️ No valid data for {trial_name} - {region} - {block}")
                continue

            # Change to 1 row and 4 columns
            fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharex=False)

            for i, disease_metric in enumerate(disease_metrics):
                ax = axes[i]
                all_dates = []

                for block_var in block_vars:
                    filtered = [
                        e for e in entries
                        if block_var in e and disease_metric in e and isinstance(e.get('Date'), datetime)
                    ]
                    if not filtered:
                        continue

                    filtered.sort(key=lambda x: x['Date'])
                    dates = [e['Date'] for e in filtered]
                    y_block = [e[block_var] for e in filtered]
                    ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
                    all_dates.extend(dates)

                disease_filtered = [
                    e for e in entries if disease_metric in e and isinstance(e.get('Date'), datetime)
                ]
                if disease_filtered:
                    disease_filtered.sort(key=lambda x: x['Date'])
                    dates_disease = [e['Date'] for e in disease_filtered]
                    y_disease = [e[disease_metric] for e in disease_filtered]

                    ax2 = ax.twinx()
                    ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=disease_metric)
                    ax2.set_ylabel(disease_metric, color='red')
                    ax2.tick_params(axis='y', labelcolor='red')
                    ax2.set_ylim(top=100)

                ax.set_title(f"{disease_metric} vs Block Vars")
                ax.set_ylabel("Block Var Value")
                ax.tick_params(axis='x', rotation=45)
                ax.grid(True)

                if all_dates:
                    ax.set_xticks(sorted(set(all_dates)))
                    ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

                ax.legend(fontsize='small', loc='upper left')

            plt.suptitle(f"{trial_name} | Region: {region} | Block: {block}", fontsize=16)
            plt.tight_layout(rect=[0, 0, 1, 0.93])
            plt.show()


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
'''
pdf_path = "all_trial_plots.pdf"
with PdfPages(pdf_path) as pdf:
    for trial_name, trial_data in split_region_data.items():
        for region in trial_data:
            for block in block_names:
                entries = split_region_data[trial_name][region][block]
                block_vars = sorted(set(
                    k for e in entries for k in e
                    if k not in disease_metrics and k != 'Date'
                ))
                if not entries or not block_vars:
                    continue

                fig, axes = plt.subplots(2, 2, figsize=(12, 9))
                axes = axes.flatten()

                for i, disease_metric in enumerate(disease_metrics):
                    ax = axes[i]
                    all_dates = []
                    for block_var in block_vars:
                        filtered = [
                            e for e in entries
                            if block_var in e and disease_metric in e and isinstance(e.get('Date'), datetime)
                        ]
                        if not filtered:
                            continue
                        filtered.sort(key=lambda x: x['Date'])
                        dates = [e['Date'] for e in filtered]
                        y_block = [e[block_var] for e in filtered]
                        ax.plot(dates, y_block, marker='o', label=block_var)
                        all_dates.extend(dates)

                    disease_filtered = [
                        e for e in entries if disease_metric in e and isinstance(e.get('Date'), datetime)
                    ]
                    if disease_filtered:
                        disease_filtered.sort(key=lambda x: x['Date'])
                        dates_disease = [e['Date'] for e in disease_filtered]
                        y_disease = [e[disease_metric] for e in disease_filtered]
                        ax2 = ax.twinx()
                        ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red')
                        ax2.set_ylabel(disease_metric, color='red')
                        ax2.tick_params(axis='y', labelcolor='red')

                    ax.set_title(f"{disease_metric} vs Block Vars")
                    ax.set_xticks(sorted(set(all_dates)))
                    ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)
                    ax.legend(fontsize='small')

                plt.suptitle(f"{trial_name} | Region: {region} | Block: {block}")
                plt.tight_layout(rect=[0, 0, 1, 0.95])
                pdf.savefig(fig)
                plt.close()
'''

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

# Disease metrics
disease_metrics = ["new_thrips", "PDI_SB_new", "PDI_PB_new", "PDI_AN_new"]
block_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'last_7_days', 'others', 'prev_day']

# Loop through each trial
for trial_name, trial_data in split_region_data.items():
    print(f"\n🔍 Plotting data for trial: {trial_name}")

    for region in trial_data:
        for block in block_names:
            entries = split_region_data[trial_name][region][block]

            block_vars = sorted(set(
                k for e in entries for k in e
                if k not in disease_metrics and k != 'Date'
            ))

            if not entries or not block_vars:
                print(f"⚠️ No valid data for {trial_name} - {region} - {block}")
                continue

            # Start a new row of 4 plots (one for each disease metric)
            fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharex=False)

            for i, disease_metric in enumerate(disease_metrics):
                ax = axes[i]
                all_dates = []

                for block_var in block_vars:
                    filtered = [
                        e for e in entries
                        if block_var in e and disease_metric in e and isinstance(e.get('Date'), datetime)
                    ]
                    if not filtered:
                        continue

                    filtered.sort(key=lambda x: x['Date'])
                    dates = [e['Date'] for e in filtered]
                    y_block = [e[block_var] for e in filtered]
                    ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
                    all_dates.extend(dates)

                disease_filtered = [
                    e for e in entries if disease_metric in e and isinstance(e.get('Date'), datetime)
                ]
                if disease_filtered:
                    disease_filtered.sort(key=lambda x: x['Date'])
                    dates_disease = [e['Date'] for e in disease_filtered]
                    y_disease = [e[disease_metric] for e in disease_filtered]

                    ax2 = ax.twinx()
                    ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=disease_metric)
                    ax2.set_ylabel(disease_metric, color='red')
                    ax2.tick_params(axis='y', labelcolor='red')
                    ax2.set_ylim(top=100)

                ax.set_title(f"{disease_metric}")
                ax.set_ylabel("Block Var Value")
                ax.tick_params(axis='x', rotation=45)
                ax.grid(True)

                if all_dates:
                    ax.set_xticks(sorted(set(all_dates)))
                    ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

                ax.legend(fontsize='small', loc='upper left')

            plt.suptitle(f"{trial_name} | Region: {region} | Block: {block}", fontsize=16)
            plt.tight_layout(rect=[0, 0, 1, 0.93])
            plt.show()


In [ ]:
# print pdf pages 
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = "all_trial_plots_4colns.pdf"
with PdfPages(pdf_path) as pdf:
    # Loop through each trial
    for trial_name, trial_data in split_region_data.items():
        print(f"\n🔍 Plotting data for trial: {trial_name}")
    
        for region in trial_data:
            for block in block_names:
                entries = split_region_data[trial_name][region][block]
    
                block_vars = sorted(set(
                    k for e in entries for k in e
                    if k not in disease_metrics and k != 'Date'
                ))
    
                if not entries or not block_vars:
                    print(f"⚠️ No valid data for {trial_name} - {region} - {block}")
                    continue
    
                # Start a new row of 4 plots (one for each disease metric)
                fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharex=False)
    
                for i, disease_metric in enumerate(disease_metrics):
                    ax = axes[i]
                    all_dates = []
    
                    for block_var in block_vars:
                        filtered = [
                            e for e in entries
                            if block_var in e and disease_metric in e and isinstance(e.get('Date'), datetime)
                        ]
                        if not filtered:
                            continue
    
                        filtered.sort(key=lambda x: x['Date'])
                        dates = [e['Date'] for e in filtered]
                        y_block = [e[block_var] for e in filtered]
                        ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
                        all_dates.extend(dates)
    
                    disease_filtered = [
                        e for e in entries if disease_metric in e and isinstance(e.get('Date'), datetime)
                    ]
                    if disease_filtered:
                        disease_filtered.sort(key=lambda x: x['Date'])
                        dates_disease = [e['Date'] for e in disease_filtered]
                        y_disease = [e[disease_metric] for e in disease_filtered]
    
                        ax2 = ax.twinx()
                        ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=disease_metric)
                        ax2.set_ylabel(disease_metric, color='red')
                        ax2.tick_params(axis='y', labelcolor='red')
                        ax2.set_ylim(top=100)
    
                    ax.set_title(f"{disease_metric}")
                    ax.set_ylabel("Block Var Value")
                    ax.tick_params(axis='x', rotation=45)
                    ax.grid(True)
    
                    if all_dates:
                        ax.set_xticks(sorted(set(all_dates)))
                        ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)
    
                    ax.legend(fontsize='small', loc='upper left')
    
                plt.suptitle(f"{trial_name} | Region: {region} | Block: {block}", fontsize=16)
                plt.tight_layout(rect=[0, 0, 1, 0.93])
                pdf.savefig(fig)
                plt.close()

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime
import math

# Parameters
target_region = "R1T4"
target_block = "B1"
target_metric = "new_thrips"

# Filter only trials with data
valid_trials = [
    trial_name for trial_name, trial_data in split_region_data.items()
    if target_region in trial_data and target_block in trial_data[target_region]
       and split_region_data[trial_name][target_region][target_block]
]

num_trials = len(valid_trials)
cols = 4
rows = math.ceil(num_trials / cols)

# Create subplots
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
fig.suptitle(f"{target_metric} plots for Region: {target_region}, Block: {target_block}", fontsize=18)

# Loop over trials and plot
for idx, trial_name in enumerate(valid_trials):
    row, col = divmod(idx, cols)
    ax = axes[row][col]

    entries = split_region_data[trial_name][target_region][target_block]
    block_vars = sorted(set(
        k for e in entries for k in e
        if k not in disease_metrics and k != 'Date'
    ))

    all_dates = []

    for block_var in block_vars:
        filtered = [
            e for e in entries
            if block_var in e and target_metric in e and isinstance(e.get('Date'), datetime)
        ]
        if not filtered:
            continue

        filtered.sort(key=lambda x: x['Date'])
        dates = [e['Date'] for e in filtered]
        y_block = [e[block_var] for e in filtered]

        ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
        all_dates.extend(dates)

    disease_filtered = [
        e for e in entries if target_metric in e and isinstance(e.get('Date'), datetime)
    ]
    if disease_filtered:
        disease_filtered.sort(key=lambda x: x['Date'])
        dates_disease = [e['Date'] for e in disease_filtered]
        y_disease = [e[target_metric] for e in disease_filtered]

        ax2 = ax.twinx()
        ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=target_metric)
        ax2.set_ylabel(target_metric, color='red')
        ax2.tick_params(axis='y', labelcolor='red')
        ax2.set_ylim(top=100)

    if all_dates:
        ax.set_xticks(sorted(set(all_dates)))
        ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

    ax.set_title(f"{trial_name}", fontsize=12)
    ax.set_ylabel("Block Var Value")
    ax.grid(True)
    ax.legend(fontsize='x-small', loc='upper left')

# Hide any empty axes
for idx in range(num_trials, rows * cols):
    row, col = divmod(idx, cols)
    fig.delaxes(axes[row][col])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime
import math

# Parameters
target_region = "R1T4"
target_metric = "new_thrips"
block_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'last_7_days', 'others', 'prev_day']

# Loop over each block
for block in block_names:
    # Collect all valid trials for this block
    valid_trials = [
        trial_name for trial_name, trial_data in split_region_data.items()
        if target_region in trial_data
        and block in trial_data[target_region]
        and trial_data[target_region][block]
    ]

    if not valid_trials:
        print(f"⚠️ No valid data for block: {block}")
        continue

    num_plots = len(valid_trials)
    cols = 4
    rows = math.ceil(num_plots / cols)

    # Create subplots for this block
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
    fig.suptitle(f"{target_metric} plots | Region: {target_region} | Block: {block}", fontsize=18)

    # Plot each trial for this block
    for idx, trial_name in enumerate(valid_trials):
        row, col = divmod(idx, cols)
        ax = axes[row][col]

        entries = split_region_data[trial_name][target_region][block]
        block_vars = sorted(set(
            k for e in entries for k in e
            if k not in disease_metrics and k != 'Date'
        ))

        all_dates = []

        for block_var in block_vars:
            filtered = [
                e for e in entries
                if block_var in e and target_metric in e and isinstance(e.get('Date'), datetime)
            ]
            if not filtered:
                continue

            filtered.sort(key=lambda x: x['Date'])
            dates = [e['Date'] for e in filtered]
            y_block = [e[block_var] for e in filtered]

            ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
            all_dates.extend(dates)

        disease_filtered = [
            e for e in entries if target_metric in e and isinstance(e.get('Date'), datetime)
        ]
        if disease_filtered:
            disease_filtered.sort(key=lambda x: x['Date'])
            dates_disease = [e['Date'] for e in disease_filtered]
            y_disease = [e[target_metric] for e in disease_filtered]

            ax2 = ax.twinx()
            ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=target_metric)
            ax2.set_ylabel(target_metric, color='red')
            ax2.tick_params(axis='y', labelcolor='red')
            ax2.set_ylim(top=100)

        if all_dates:
            ax.set_xticks(sorted(set(all_dates)))
            ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

        ax.set_title(f"{trial_name}", fontsize=11)
        ax.set_ylabel("Block Var Value")
        ax.grid(True)
        ax.legend(fontsize='x-small', loc='upper left')

    # Hide empty axes if needed
    for idx in range(num_plots, rows * cols):
        row, col = divmod(idx, cols)
        fig.delaxes(axes[row][col])

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
# Main code
import matplotlib.pyplot as plt
from datetime import datetime
import math

# Parameters
target_metric = "new_thrips"
region_list = ['R1T4', 'R2T4', 'R3T4']
block_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'last_7_days', 'others', 'prev_day']


for target_region in region_list:
    print(f"\n📍 Processing region: {target_region}")

    for block in block_names:
        # Collect all valid trials for this region-block pair
        valid_trials = [
            trial_name for trial_name, trial_data in split_region_data.items()
            if target_region in trial_data
            and block in trial_data[target_region]
            and trial_data[target_region][block]
        ]

        if not valid_trials:
            print(f"⚠️ No valid data for {target_region} - {block}")
            continue

        num_plots = len(valid_trials)
        cols = 4
        rows = math.ceil(num_plots / cols)

        # Create subplots for this block
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
        fig.suptitle(f"{target_metric} | Region: {target_region} | Block: {block}", fontsize=18)

        for idx, trial_name in enumerate(valid_trials):
            row, col = divmod(idx, cols)
            ax = axes[row][col]

            entries = split_region_data[trial_name][target_region][block]
            block_vars = sorted(set(
                k for e in entries for k in e
                if k not in disease_metrics and k != 'Date'
            ))

            all_dates = []

            for block_var in block_vars:
                filtered = [
                    e for e in entries
                    if block_var in e and target_metric in e and isinstance(e.get('Date'), datetime)
                ]
                if not filtered:
                    continue

                filtered.sort(key=lambda x: x['Date'])
                dates = [e['Date'] for e in filtered]
                y_block = [e[block_var] for e in filtered]

                ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
                all_dates.extend(dates)

            disease_filtered = [
                e for e in entries if target_metric in e and isinstance(e.get('Date'), datetime)
            ]
            if disease_filtered:
                disease_filtered.sort(key=lambda x: x['Date'])
                dates_disease = [e['Date'] for e in disease_filtered]
                y_disease = [e[target_metric] for e in disease_filtered]

                ax2 = ax.twinx()
                ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=target_metric)
                ax2.set_ylabel(target_metric, color='red')
                ax2.tick_params(axis='y', labelcolor='red')
                ax2.set_ylim(top=100)

            if all_dates:
                ax.set_xticks(sorted(set(all_dates)))
                ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

            ax.set_title(f"{trial_name}", fontsize=11)
            ax.set_ylabel("Block Var Value")
            ax.grid(True)
            ax.legend(fontsize='x-small', loc='upper left')

        # Hide any unused axes
        for idx in range(num_plots, rows * cols):
            row, col = divmod(idx, cols)
            fig.delaxes(axes[row][col])

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()


In [ ]:
# Main code - commented Saving to PDF

import matplotlib.pyplot as plt
from datetime import datetime
import math

# Parameters
region_list = ['R1T4', 'R2T4', 'R3T4']
block_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'last_7_days', 'others', 'prev_day']
target_metrics = ['PDI_SB_new', 'new_thrips', 'PDI_PB_new', 'PDI_AN_new']

#from matplotlib.backends.backend_pdf import PdfPages

#pdf_path = "Disease-Region-Block-Trials.pdf"
#with PdfPages(pdf_path) as pdf:
for target_metric in target_metrics:
    print(f"\n=====================\n🎯 Plotting metric: {target_metric}\n=====================")
    
    for target_region in region_list:
        print(f"\n📍 Region: {target_region}")

        for block in block_names:
            # Get trials with data for this region-block
            valid_trials = [
                trial_name for trial_name, trial_data in split_region_data.items()
                if target_region in trial_data
                and block in trial_data[target_region]
                and trial_data[target_region][block]
            ]

            if not valid_trials:
                print(f"⚠️ No data for {target_region} - {block}")
                continue

            num_plots = len(valid_trials)
            cols = 4
            rows = math.ceil(num_plots / cols)

            fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
            fig.suptitle(f"{target_metric} | Region: {target_region} | Block: {block}", fontsize=18)

            for idx, trial_name in enumerate(valid_trials):
                row, col = divmod(idx, cols)
                ax = axes[row][col]

                entries = split_region_data[trial_name][target_region][block]
                block_vars = sorted(set(
                    k for e in entries for k in e
                    if k not in target_metrics and k != 'Date'
                ))

                all_dates = []

                for block_var in block_vars:
                    filtered = [
                        e for e in entries
                        if block_var in e and target_metric in e and isinstance(e.get('Date'), datetime)
                    ]
                    if not filtered:
                        continue

                    filtered.sort(key=lambda x: x['Date'])
                    dates = [e['Date'] for e in filtered]
                    y_block = [e[block_var] for e in filtered]

                    ax.plot(dates, y_block, marker='o', linestyle='-', label=block_var)
                    all_dates.extend(dates)

                disease_filtered = [
                    e for e in entries if target_metric in e and isinstance(e.get('Date'), datetime)
                ]
                if disease_filtered:
                    disease_filtered.sort(key=lambda x: x['Date'])
                    dates_disease = [e['Date'] for e in disease_filtered]
                    y_disease = [e[target_metric] for e in disease_filtered]

                    ax2 = ax.twinx()
                    ax2.plot(dates_disease, y_disease, marker='x', linestyle='--', color='red', label=target_metric)
                    ax2.set_ylabel(target_metric, color='red')
                    ax2.tick_params(axis='y', labelcolor='red')
                    ax2.set_ylim(top=100)

                if all_dates:
                    ax.set_xticks(sorted(set(all_dates)))
                    ax.set_xticklabels([d.strftime('%d-%b') for d in sorted(set(all_dates))], rotation=45)

                ax.set_title(f"{trial_name}", fontsize=11)
                ax.set_ylabel("Block Var Value")
                ax.grid(True)
                ax.legend(fontsize='x-small', loc='upper left')

            # Hide any empty axes
            for idx in range(num_plots, rows * cols):
                row, col = divmod(idx, cols)
                fig.delaxes(axes[row][col])

            plt.tight_layout(rect=[0, 0, 1, 0.95])
            #pdf.savefig(fig)
            #plt.close()
            plt.show()
